# 🌍 Our World in Data (OWID) Health Data Access

This notebook demonstrates how to access comprehensive health datasets from **Our World in Data (OWID)**, a scientific online publication that focuses on large global problems such as poverty, disease, hunger, climate change, war, existential risks, and inequality.

## Overview

OWID provides free, open-access data covering:
- **192 countries** worldwide
- **COVID-19 data**: Cases, deaths, testing, hospitalizations, vaccinations
- **Excess mortality**: Estimates of deaths above historical averages
- **Health metrics**: Life expectancy, disease burden, risk factors
- **Policy responses**: Government stringency indices

### Data Available
- Daily updated COVID-19 statistics
- Vaccination progress by country
- Hospital and ICU capacity data
- Excess mortality estimates
- Testing data and positivity rates

**Website:** https://ourworldindata.org/health

**GitHub:** https://github.com/owid/covid-19-data

**License:** CC BY (Creative Commons Attribution)

## 📦 Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add scripts directory to path
sys.path.insert(0, str(Path.cwd().parent.parent / 'scripts'))
%matplotlib inline
from accessors import OWIDAccessor

# Initialize accessor
owid = OWIDAccessor()

print("✅ OWID Accessor initialized successfully!")

## 🌍 Exploring OWID Coverage

In [ ]:
# List all available countries
countries = owid.list_countries()
print(f"🌍 Total countries available: {len(countries)}")
print("\nFirst 15 countries:")
countries.head(15)

In [ ]:
# List regions
regions = owid.list_regions()
print("📍 Countries by Region:")
for region in regions['region'].unique():
    n_countries = len(regions[regions['region'] == region])
    print(f"\n{region}: {n_countries} countries")
    # Show first 5 countries in each region
    sample = regions[regions['region'] == region]['country_code'].head(5).tolist()
    print(f"  Sample: {', '.join(sample)}")

In [ ]:
# List available datasets
datasets = owid.list_available_datasets()
print(f"📊 Available Datasets: {len(datasets)}")
datasets

## 🦠 COVID-19 Data

In [ ]:
# Get COVID-19 data for selected countries
covid_data = owid.get_covid_data(
    countries=['BRA', 'USA', 'IND', 'GBR', 'ITA'],
    metrics=['cases', 'deaths', 'hospitalizations'],
    start_date='2021-01-01',
    end_date='2021-12-31'
)

print(f"Retrieved {len(covid_data)} records")
print("\nSample data:")
covid_data.head(10)

In [ ]:
# Get latest data for all countries
latest = owid.get_latest_data(countries=['BRA', 'USA', 'IND', 'GBR', 'ITA'])

# Display key metrics
if not latest.empty:
    columns_to_show = ['country_code', 'date', 'total_cases', 'total_deaths', 'total_vaccinations']
    available_cols = [c for c in columns_to_show if c in latest.columns]
    print("\nLatest COVID-19 Statistics:")
    print(latest[available_cols])

In [ ]:
# Visualize COVID-19 cases over time
if not covid_data.empty and 'new_cases_smoothed' in covid_data.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for country in covid_data['country_code'].unique():
        country_data = covid_data[covid_data['country_code'] == country]
        if 'new_cases_smoothed' in country_data.columns:
            ax.plot(country_data['date'], country_data['new_cases_smoothed'], 
                   label=country, linewidth=2)
    
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('New Cases (7-day smoothed)', fontsize=12)
    ax.set_title('COVID-19 Daily New Cases - 2021', fontsize=14)
    ax.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 💉 Vaccination Data

In [ ]:
# Get vaccination data
vax_data = owid.get_vaccination_data(
    countries=['BRA', 'USA', 'GBR', 'ISR', 'CHL', 'ARE'],
    start_date='2021-01-01'
)

print(f"Retrieved {len(vax_data)} vaccination records")

if not vax_data.empty:
    # Show latest vaccination rates
    latest_vax = vax_data.sort_values('date').groupby('country_code').last().reset_index()
    cols = ['country_code', 'date', 'people_fully_vaccinated_per_hundred']
    available_cols = [c for c in cols if c in latest_vax.columns]
    print("\nLatest Full Vaccination Rates (%):")
    print(latest_vax[available_cols])

In [ ]:
# Visualize vaccination progress
if not vax_data.empty and 'people_fully_vaccinated_per_hundred' in vax_data.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for country in vax_data['country_code'].unique():
        country_data = vax_data[vax_data['country_code'] == country]
        ax.plot(country_data['date'], country_data['people_fully_vaccinated_per_hundred'], 
               label=country, linewidth=2, marker='o', markersize=3)
    
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Fully Vaccinated (% of population)', fontsize=12)
    ax.set_title('COVID-19 Vaccination Progress', fontsize=14)
    ax.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.show()

## 📊 Excess Mortality Analysis

In [ ]:
# Get excess mortality data
excess = owid.get_excess_mortality(
    countries=['USA', 'GBR', 'ITA', 'ESP', 'BRA', 'IND'],
    start_date='2020-03-01'
)

print(f"Retrieved {len(excess)} excess mortality records")
if not excess.empty:
    print("\nColumns available:", excess.columns.tolist())
    print("\nSample data:")
    excess.head(10)

In [ ]:
# Calculate cumulative excess deaths by country
if not excess.empty and 'excess_deaths' in excess.columns:
    total_excess = excess.groupby('country_code')['excess_decess_deaths'].sum().sort_values(ascending=False)
    print("\nTotal Excess Deaths by Country:")
    for country, deaths in total_excess.items():
        print(f"  {country}: {deaths:,.0f}")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    total_excess.plot(kind='bar', ax=ax, color='coral')
    ax.set_xlabel('Country', fontsize=12)
    ax.set_ylabel('Total Excess Deaths', fontsize=12)
    ax.set_title('COVID-19 Excess Mortality by Country', fontsize=14)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Format y-axis with thousand separators
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
    
    plt.tight_layout()
    plt.show()

## 🏥 Testing Data

In [ ]:
# Get COVID-19 testing data
testing = owid.get_testing_data(
    countries=['BRA', 'USA', 'GBR', 'DEU', 'KOR'],
    start_date='2021-01-01',
    end_date='2021-06-30'
)

print(f"Retrieved {len(testing)} testing records")
if not testing.empty:
    print("\nSample data:")
    cols = ['country_code', 'date', 'total_tests', 'new_tests', 'positive_rate']
    available_cols = [c for c in cols if c in testing.columns]
    print(testing[available_cols].head(10))

In [ ]:
# Visualize test positivity rate
if not testing.empty and 'positive_rate' in testing.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for country in testing['country_code'].unique():
        country_data = testing[testing['country_code'] == country]
        ax.plot(country_data['date'], country_data['positive_rate'] * 100, 
               label=country, linewidth=2, marker='o', markersize=3)
    
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Test Positivity Rate (%)', fontsize=12)
    ax.set_title('COVID-19 Test Positivity Rate (H1 2021)', fontsize=14)
    ax.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=5, color='r', linestyle='--', alpha=0.5, label='5% threshold')
    
    plt.tight_layout()
    plt.show()

## 🌐 Global Summary

In [ ]:
# Get global summary statistics
summary = owid.get_global_summary()
print("🌐 Global COVID-19 Summary:")
print("=" * 50)
for _, row in summary.iterrows():
    value = row['value']
    if value >= 1e9:
        formatted = f"{value/1e9:.2f}B"
    elif value >= 1e6:
        formatted = f"{value/1e6:.2f}M"
    elif value >= 1e3:
        formatted = f"{value/1e3:.2f}K"
    else:
        formatted = f"{value:.0f}"
    print(f"  {row['metric']}: {formatted} {row['unit']}")

## 🔍 Country Comparison

In [ ]:
# Compare countries on different metrics
countries_to_compare = ['BRA', 'USA', 'IND', 'GBR', 'ITA', 'FRA', 'DEU', 'JPN']

# Compare total cases per million
comparison_cases = owid.compare_countries(
    countries=countries_to_compare,
    metric='total_cases_per_million'
)

print("📊 Comparison: Total Cases per Million Population")
print(comparison_cases.tail(1).T)  # Show latest values

In [ ]:
# Compare total deaths per million
comparison_deaths = owid.compare_countries(
    countries=countries_to_compare,
    metric='total_deaths_per_million'
)

print("📊 Comparison: Total Deaths per Million Population")
print(comparison_deaths.tail(1).T)  # Show latest values

In [ ]:
# Visualize comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot cases per million
if not comparison_cases.empty:
    for country in comparison_cases.columns:
        ax1.plot(comparison_cases.index, comparison_cases[country], label=country, linewidth=2)
    ax1.set_title('Total Cases per Million', fontsize=14)
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Cases per Million')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

# Plot deaths per million
if not comparison_deaths.empty:
    for country in comparison_deaths.columns:
        ax2.plot(comparison_deaths.index, comparison_deaths[country], label=country, linewidth=2)
    ax2.set_title('Total Deaths per Million', fontsize=14)
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Deaths per Million')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🌎 Regional Aggregates

In [ ]:
# Get regional aggregates
regions_to_analyze = ['South America', 'North America', 'Europe', 'Asia']

fig, ax = plt.subplots(figsize=(14, 6))

for region in regions_to_analyze:
    try:
        region_data = owid.get_region_aggregates(
            region=region,
            metric='new_cases',
            aggregation='sum'
        )
        
        if not region_data.empty:
            ax.plot(region_data['date'], region_data['new_cases'], 
                   label=region, linewidth=2)
    except Exception as e:
        print(f"Could not fetch data for {region}: {e}")

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('New Cases', fontsize=12)
ax.set_title('COVID-19 Daily New Cases by Region', fontsize=14)
ax.legend(title='Region')
ax.grid(True, alpha=0.3)

# Format y-axis
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))

plt.tight_layout()
plt.show()

## 📈 Hospitalizations Data

In [ ]:
# Get hospitalization data
hosp_data = owid.get_hospitalizations_data(
    countries=['USA', 'GBR', 'ITA', 'ESP', 'FRA'],
    start_date='2021-01-01',
    end_date='2021-12-31'
)

print(f"Retrieved {len(hosp_data)} hospitalization records")
if not hosp_data.empty:
    print("\nColumns:", hosp_data.columns.tolist())
    print("\nSample data:")
    hosp_data.head()

In [ ]:
# Visualize ICU patients
if not hosp_data.empty and 'icu_patients' in hosp_data.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for country in hosp_data['country_code'].unique():
        country_data = hosp_data[hosp_data['country_code'] == country]
        ax.plot(country_data['date'], country_data['icu_patients'], 
               label=country, linewidth=2)
    
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('ICU Patients', fontsize=12)
    ax.set_title('COVID-19 ICU Patients by Country (2021)', fontsize=14)
    ax.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    
    # Format y-axis
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e3:.0f}K' if x >= 1e3 else f'{x:.0f}'))
    
    plt.tight_layout()
    plt.show()

## 🔧 Advanced Usage

In [ ]:
# Get multiple metrics at once
multi_metric_data = owid.get_covid_data(
    countries=['BRA', 'USA'],
    metrics=['cases', 'deaths', 'tests', 'hospitalizations', 'reproduction_rate'],
    start_date='2021-06-01',
    end_date='2021-09-30'
)

print(f"Retrieved {len(multi_metric_data)} records with multiple metrics")
print("\nAvailable columns:")
for col in multi_metric_data.columns:
    print(f"  - {col}")

In [ ]:
# Analyze R (reproduction) rate
if not multi_metric_data.empty and 'reproduction_rate' in multi_metric_data.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for country in multi_metric_data['country_code'].unique():
        country_data = multi_metric_data[multi_metric_data['country_code'] == country]
        ax.plot(country_data['date'], country_data['reproduction_rate'], 
               label=country, linewidth=2, marker='o', markersize=4)
    
    # Add reference line at R=1
    ax.axhline(y=1, color='r', linestyle='--', alpha=0.7, label='R=1 threshold')
    
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Reproduction Rate (R)', fontsize=12)
    ax.set_title('COVID-19 Reproduction Rate (R) - Q3 2021', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 3)
    
    plt.tight_layout()
    plt.show()

## 📚 References

- **Our World in Data - Health:** https://ourworldindata.org/health
- **COVID-19 Data:** https://github.com/owid/covid-19-data
- **General Datasets:** https://github.com/owid/owid-datasets
- **Data API:** https://covid.ourworldindata.org/data
- **License:** CC BY (Creative Commons Attribution)

## 📝 Notes

1. **Data Updates:** COVID-19 data is updated daily, other datasets weekly.
2. **Country Codes:** Use ISO3 codes (e.g., 'BRA' for Brazil, 'USA' for United States).
3. **Missing Data:** Some countries may have incomplete data for certain metrics.
4. **Rate Limiting:** The accessor includes automatic retry logic for API requests.
5. **Caching:** Consider caching results for repeated analysis of the same data.